# 08P: Physical-object mapping and freeze

Reuse the sealed 08N/08O features and frozen object-bank scorer. Compare one joint assignment with the existing all-candidate map and winner-consensus baseline on the same 100 development scenes. Supplied correspondence remains a separately labelled diagnostic. No new model inference is needed.

## 1. Configure the local cached run

Set `VPI_OBJECT_BANK_CONFIG` to the private mapping configuration. Scientific outputs are separate from reports and tracking, and each scene is checkpointed. One, two, three and five views use only their declared images.

In [ ]:
import os, json, sys
from pathlib import Path
ROOT = Path(os.environ['VPI_OBJECT_BANK_ROOT'])
sys.path.insert(0, str(ROOT / 'src'))
cfg = json.loads(Path(os.environ['VPI_OBJECT_BANK_CONFIG']).read_text())
from vpi_object_mapping.protocol import PROTOCOL
from vpi_object_bank.storage import read_json, verify_seal
print(json.dumps(PROTOCOL, indent=2))


## 2. Preserve physical-object hypotheses

The matcher maximises supported appearance links jointly. Every candidate belongs to one track, with at most one observation per uploaded view. Unmatched detections remain explicit singleton hypotheses. Up to three globally ranked maps expose ambiguity; the primary ranking uses only the best map. Solver optimality concerns this appearance objective, not correctness of physical identity.

In [ ]:
from vpi_object_mapping.prediction import predict
prediction_receipt = predict(cfg, ROOT)
print(json.dumps(prediction_receipt, indent=2))
assert prediction_receipt['sealed']
verify_seal(cfg['prediction_root'], name='prediction_lock.json')


## 3. Evaluate sealed predictions

Only now does the evaluator open scene labels. It checks exact reuse of the earlier baselines, candidate conservation and selected-view membership. Denominators stay at 100 scenes; confidence intervals resample asset groups.

In [ ]:
from vpi_object_mapping.evaluation import evaluate
evaluation_receipt = evaluate(cfg['prediction_root'], cfg['previous_prediction_root'], cfg['prepared_root'], cfg['scoring_root'], cfg['evaluator_root'], cfg['evaluation_root'], ROOT)
print(json.dumps(evaluation_receipt, indent=2))


## 4. Decide once

Promotion requires an improvement over winner consensus under the registered primary-three-view rule, alongside identity and fragmentation checks. A failed gate ends this matcher/scorer search. It does not remove difficult scenes or trigger a new weight search.

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv(Path(cfg['evaluation_root']) / 'shareable/view_schedule_summary.csv')
primary = summary[(summary.metric == 'top1_tie_expected') & summary.view_set_id.isin(['two-primary','three-primary','five-primary'])]
display(primary[['method','view_count','mean','ci_low','ci_high','scene_count']])
print(json.dumps(read_json(Path(cfg['evaluation_root']) / 'shareable/decision.json'), indent=2))


## 5. Inspect supporting evidence

The aggregate figure averages schedules within each scene first. Private galleries retain original views, consistent proposed IDs, saved mask-bounding boxes, masked objects, coarse patch evidence and combined scores. Examples include a fixed first scene plus explicitly post-hoc high/low accuracy illustrations. These examples never determine evaluation membership.

In [ ]:
from vpi_object_mapping.reporting import build_report
report_receipt = build_report(cfg, ROOT)
print(json.dumps(report_receipt, indent=2))
display(Image(filename=str(Path(cfg['report_root']) / 'shareable/figures/view_count_accuracy.png')))
display(Image(filename=str(Path(cfg['report_root']) / 'shareable/figures/paired_improvement.png')))


In [ ]:
gallery_manifest = read_json(Path(cfg['report_root']) / 'private/gallery_manifest.json')
for item in gallery_manifest:
    print(item['selection_reason'], item['mapping_status'], item['uncertainty']['status'])
    display(Image(filename=str(Path(cfg['report_root']) / 'private' / item['file'])))


## 6. Publish aggregate tracking after the scientific seal

Tracking is bounded and recoverable. Credentials come from the normal local configuration. Private scene records and gallery images are excluded. A publication failure preserves the experiment and can be retried without recomputation.

In [ ]:
import subprocess
receipt_path = ROOT / '.local/object-mapping/tracking/receipt.json'
command = [sys.executable, str(ROOT / 'scripts/publish_clearml.py'), '--scientific-root', cfg['evaluation_root'], '--shareable-root', str(Path(cfg['evaluation_root']) / 'shareable'), '--report-root', cfg['report_root'], '--receipt', str(receipt_path), '--task-name', '08P Physical object mapping and freeze', '--bounded', '--timeout', '180']
try:
    publication = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=210)
    publication_exit = publication.returncode
except subprocess.TimeoutExpired:
    from vpi_object_bank.storage import atomic_json
    atomic_json(receipt_path.with_name('outer_timeout.json'), {'status':'tracking_pending', 'reason':'outer_process_timeout', 'scientific_results_preserved':True})
    publication_exit = None
if receipt_path.exists():
    receipt = read_json(receipt_path)
    print({key:receipt.get(key) for key in ['status','task_id','task_url','uploaded_artifact_count','remote_artifacts_verified']})
else:
    print({'status':'tracking_pending','exit_code':publication_exit})


## Method basis and scope

Joint map consistency is motivated by [multi-way matching](https://proceedings.neurips.cc/paper/2013/hash/3df1d4b96d8976ff5986393e8767f5b2-Abstract.html). The small set-partition objective here is our bounded adaptation, not an implementation of that paper. [SciPy MILP](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) provides explicit optimality and time-limit status. No 3D geometry, pose invariance, calibrated identity probability or independent test accuracy is claimed.